In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import pickle
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import yaml
import matplotlib.dates as mdates

from neuralhydrology.modelzoo import get_model
from neuralhydrology.utils.config import Config
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.customlstm import CustomLSTM

In [2]:
# ------- Paths -------
model_dir=Path('../final_runs/runs/precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2302_000856')
pickle_file_dir = model_dir / 'validation' / 'model_epoch030' / 'validation_results.p'
pt_file_dir = model_dir / 'model_epoch030.pt'
optimizer_file_dir = model_dir / 'optimizer_state_epoch030.pt'
scaler_path = model_dir / "train_data" / "train_data_scaler.yml"
esp_hydrographs_output_dir = Path("./hydrographs_esp/precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2302_000856")
attributes_file = Path("../final_runs/data/attributes/attributes.csv")
ts_file = Path("../final_runs/data/time_series")

In [3]:
pt_data = torch.load(pt_file_dir,weights_only=True)
# pt_data

In [4]:
cfg = Config(model_dir / "config.yml")

In [5]:
optimized_model = CudaLSTM(cfg)
# print(optimized_model)

state_dict = torch.load(pt_file_dir)

optimized_model.load_state_dict(state_dict)
optimized_model.eval()

/tmp/ipykernel_44791/2693144925.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pt_file_dir)


CudaLSTM(
  (embedding_net): InputLayer(
    (statics_embedding): Identity()
    (dynamics_embeddings): ModuleList(
      (0): Identity()
    )
  )
  (lstm): LSTM(18, 256)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Regression(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=1, bias=True)
    )
  )
)

In [10]:
# SCALER
with open(scaler_path, "r") as f:
    scaler = yaml.safe_load(f)

In [6]:
# DYNAMIC INPUTS
def norm_dyn(tensor, varname):
    center = scaler["xarray_feature_center"]["data_vars"][varname]["data"]
    scale  = scaler["xarray_feature_scale"]["data_vars"][varname]["data"]
    center = torch.tensor(center, dtype=tensor.dtype, device=tensor.device)
    scale  = torch.tensor(scale,  dtype=tensor.dtype, device=tensor.device)
    return (tensor - center) / scale

In [7]:
# custom_lstm = CustomLSTM(cfg=cfg)

# custom_lstm.copy_weights(optimized_model)

# custom_lstm.eval()

# # SCALER
# with open(scaler_path, "r") as f:
#     scaler = yaml.safe_load(f)

# # LOADING ATTRIBUTES
# attributes_file = Path("../CHIRPS_2.0/filtered_data_gauge_and_CHIRPS/attributes/attributes.csv")
# df_attr = pd.read_csv(attributes_file, index_col=0)

# # --- Define the features your model uses ---
# static_features = [
#     "elev_mean",
#     "slope_mean",
#     "area_gages2",
#     "sand_frac",
#     "silt_frac",
#     "clay_frac",
#     "p_mean",
#     "pet_mean",
#     "aridity",
#     "high_prec_dur",
#     "low_prec_dur"
# ]

# # Ensure the index is a string
# df_attr.index = df_attr.index.astype(str)

# # Select your basin
# basin = "CAMELS_UY_6"

# # static_values
# static_values = df_attr.loc[basin, static_features].to_numpy(dtype=np.float32)

# attr_means = np.array([scaler["attribute_means"][k] for k in static_features], dtype=np.float32)
# attr_stds  = np.array([scaler["attribute_stds"][k]  for k in static_features], dtype=np.float32)

# static_norm = (static_values - attr_means) / attr_stds
# static_tensor = torch.tensor(static_norm).unsqueeze(0)  # [1, n_static]

# #Timeseries
# ts_file = Path("../CHIRPS_2.0/filtered_data_gauge_and_CHIRPS/time_series") / f"{basin}.nc"
# ds = xr.open_dataset(ts_file)

# dynamic_vars = ["QObs_mm_d","prcp_mm_day","srad_W_m2","tmax_C","tmin_C","prcp_chirps_mm_day"]

# # --- Convert xarray to DataFrame and sort by date ---
# df_dyn = ds[dynamic_vars].to_dataframe().sort_index()

# spinup_start = pd.to_datetime("1989-10-01")
# spinup_end   = pd.to_datetime("2008-09-30")

# spinup_df = df_dyn.loc[spinup_start:spinup_end][dynamic_vars]

# # --- Convert to tensor ---
# spinup_tensor = torch.tensor(spinup_df.values.astype(np.float32)).unsqueeze(0)

# # historical_dynamic_tensor: [1, 270, 6] as you already computed
# x_prcp      = spinup_tensor[..., 1:2]
# x_srad      = spinup_tensor[..., 2:3]
# x_tmax      = spinup_tensor[..., 3:4]
# x_tmin      = spinup_tensor[..., 4:5]
# x_prcp_ch   = spinup_tensor[..., 5:6]

# x_prcp_norm    = norm_dyn(x_prcp,    "prcp_mm_day")
# x_srad_norm    = norm_dyn(x_srad,    "srad_W_m2")
# x_tmax_norm    = norm_dyn(x_tmax,    "tmax_C")
# x_tmin_norm    = norm_dyn(x_tmin,    "tmin_C")
# x_prcp_ch_norm = norm_dyn(x_prcp_ch, "prcp_chirps_mm_day")

# spinup_inputs = {
#     "x_d": {
#         "prcp_mm_day":        x_prcp_norm,
#         "srad_W_m2":          x_srad_norm,
#         "tmax_C":             x_tmax_norm,
#         "tmin_C":             x_tmin_norm,
#         "prcp_chirps_mm_day": x_prcp_ch_norm,
#     },
#     "x_s": static_tensor
# }

# spinup_inputs = {
#     "x_d": {k: v for k, v in spinup_inputs["x_d"].items()},
#     "x_s": spinup_inputs["x_s"]
# }

# with torch.no_grad():
#     out = custom_lstm(spinup_inputs)

# # h0 = out["h_n"]
# # c0 = out["c_n"]
# h_complete_run = out["h_n"][:, -1, :]
# c_complete_run = out["c_n"][:, -1, :]
# y_hat = out["y_hat"]

# prediction_start = pd.to_datetime("2008-10-01")
# prediction_end   = pd.to_datetime("2008-12-31")

# predictions_dict = {
#     "year": [],
#     "prediction": []
# }  

# for year in range(1989, 2008):
#     start = pd.to_datetime(f"{year}-10-01")
#     end   = pd.to_datetime(f"{year}-12-31")

#     historical_df = df_dyn.loc[start:end][dynamic_vars]

#     historical_df.index = historical_df.index.map(lambda d: d.replace(year=2008))

#     historical_tensor = torch.tensor(historical_df.values.astype(np.float32)).unsqueeze(0)

#     # historical_dynamic_tensor: [1, 270, 6] as you already computed
#     x_prcp_h      = historical_tensor[..., 1:2]
#     x_srad_h      = historical_tensor[..., 2:3]
#     x_tmax_h      = historical_tensor[..., 3:4]
#     x_tmin_h      = historical_tensor[..., 4:5]
#     x_prcp_ch_h   = historical_tensor[..., 5:6]

#     x_prcp_norm_h    = norm_dyn(x_prcp_h,    "prcp_mm_day")
#     x_srad_norm_h    = norm_dyn(x_srad_h,    "srad_W_m2")
#     x_tmax_norm_h    = norm_dyn(x_tmax_h,    "tmax_C")
#     x_tmin_norm_h    = norm_dyn(x_tmin_h,    "tmin_C")
#     x_prcp_ch_norm_h = norm_dyn(x_prcp_ch_h, "prcp_chirps_mm_day")

#     historical_inputs = {
#         "x_d": {
#             "prcp_mm_day":        x_prcp_norm_h,
#             "srad_W_m2":          x_srad_norm_h,
#             "tmax_C":             x_tmax_norm_h,
#             "tmin_C":             x_tmin_norm_h,
#             "prcp_chirps_mm_day": x_prcp_ch_norm_h,
#         },
#         "x_s": static_tensor
#     }

#     historical_inputs = {
#         "x_d": {k: v for k, v in historical_inputs["x_d"].items()},
#         "x_s": historical_inputs["x_s"]
#     }

#     with torch.no_grad():
#         out = custom_lstm(historical_inputs, h_0=h_complete_run, c_0=c_complete_run) #h_val and c_val come from the spin up phase

#     #predictions
#     y_hat = out["y_hat"]

#     # Denormalize predictions
#     q_center = scaler["xarray_feature_center"]["data_vars"]["QObs_mm_d"]["data"]
#     q_scale  = scaler["xarray_feature_scale"]["data_vars"]["QObs_mm_d"]["data"]

#     q_center = torch.tensor(q_center, dtype=y_hat.dtype, device=y_hat.device)
#     q_scale  = torch.tensor(q_scale,  dtype=y_hat.dtype, device=y_hat.device)

#     y_hat_denorm = y_hat * q_scale + q_center

#     # Store predictions in the dictionary
#     predictions_dict["year"].append(year)         
#     predictions_dict["prediction"].append(y_hat_denorm[0, :, 0].cpu().numpy())

# obs_3_months = []

# for year in range(1989, 2008):
#     obs_vals = spinup_df.loc[f"{year}-10-01":f"{year}-12-31", 'QObs_mm_d'].values
#     obs_3_months.append(obs_vals)

# fig, ax = plt.subplots(figsize=(12, 5))

# # Fixed date range for x-axis
# dates = pd.date_range(start="2008-10-01", end="2008-12-31")

# # Stack all predictions into a 2D array: [n_years, n_days]
# all_preds = np.array(predictions_dict["prediction"])  # shape: (20, n_days)

# # Compute median across years (axis=0)
# median_pred = np.median(all_preds, axis=0)
# percentile_10 = np.percentile(all_preds, 10, axis=0)
# percentile_90 = np.percentile(all_preds, 90, axis=0)

# # Observed metrics (using nan-aware functions)
# all_obs = np.array(obs_3_months)  # shape: (n_years, 92)
# median_obs = np.nanmedian(all_obs, axis=0)  # shape: (92,)
# percentile_10_obs = np.nanpercentile(all_obs, 10, axis=0)
# percentile_90_obs = np.nanpercentile(all_obs, 90, axis=0)

# # Shade between 10th and 90th percentile
# ax.fill_between(dates, percentile_10_obs, percentile_90_obs, color="blue", alpha=0.2, label="10th-90th Percentile (Obs)")
# ax.fill_between(dates, percentile_10, percentile_90, color="red", alpha=0.2, label="10th-90th Percentile (Pred)")

# # Plot median
# ax.plot(dates, median_obs, color="blue", label="Median Observed (Obs)", linewidth=2)
# ax.plot(dates, median_pred, color="red", label="Median Prediction (Pred)", linewidth=2)

# ax.set_xlabel("Date")
# ax.set_ylabel("Streamflow (mm/d)")
# ax.set_title("Median Predicted Streamflow")
# ax.legend()
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))  # 'Oct 01', 'Nov 15', etc.
# # plt.show()

# fig.savefig(esp_hydrographs_output_dir / f"{basin}_hydrograph.png", dpi=300)

In [8]:
basins= ["CAMELS_UY_2",
    "CAMELS_UY_3",
    "CAMELS_UY_5",
    "CAMELS_UY_6",
    "CAMELS_UY_7",
    "CAMELS_UY_8",
    "CAMELS_UY_9",
    "CAMELS_UY_10",
    "CAMELS_UY_11",
    "CAMELS_UY_15",
    "CAMELS_UY_16"]

In [13]:
for basin in basins:
    for month in range(1, 13):
        custom_lstm = CustomLSTM(cfg=cfg)

        custom_lstm.copy_weights(optimized_model)

        custom_lstm.eval()

        # # SCALER
        # with open(scaler_path, "r") as f:
        #     scaler = yaml.safe_load(f)

        # LOADING ATTRIBUTES
        df_attr = pd.read_csv(attributes_file, index_col=0)

        # --- Define the features your model uses ---
        static_features = [
            "elev_mean",
            "slope_mean",
            "area_gages2",
            "sand_frac",
            "silt_frac",
            "clay_frac",
            "p_mean",
            "pet_mean",
            "aridity",
            "high_prec_dur",
            "low_prec_dur"
        ]

        # Ensure the index is a string
        df_attr.index = df_attr.index.astype(str)

        # # Select your basin
        # basin = "CAMELS_UY_6"

        # static_values
        static_values = df_attr.loc[basin, static_features].to_numpy(dtype=np.float32)

        attr_means = np.array([scaler["attribute_means"][k] for k in static_features], dtype=np.float32)
        attr_stds  = np.array([scaler["attribute_stds"][k]  for k in static_features], dtype=np.float32)

        static_norm = (static_values - attr_means) / attr_stds
        static_tensor = torch.tensor(static_norm).unsqueeze(0)  # [1, n_static]

        #Timeseries
        ts_file_dir = ts_file / f"{basin}.nc"
        ds = xr.open_dataset(ts_file_dir)

        dynamic_vars = ["QObs_mm_d","srad_W_m2","tmax_C","tmin_C","prcp_mm_day","prcp_chirps_mm_day","prcp_mswep_mm_day","prcp_gauge_mm_day"]

        # --- Convert xarray to DataFrame and sort by date ---
        df_dyn = ds[dynamic_vars].to_dataframe().sort_index()

        spinup_start = pd.to_datetime("1989-10-01")
        prediction_start = pd.Timestamp(year=2008, month=month, day=1)
        spinup_end = prediction_start - pd.Timedelta(days=1)
        prediction_end = prediction_start + pd.DateOffset(months=3) - pd.Timedelta(days=1) # 3-month prediction window

        spinup_df = df_dyn.loc[spinup_start:spinup_end][dynamic_vars]

        # --- Convert to tensor ---
        spinup_tensor = torch.tensor(spinup_df.values.astype(np.float32)).unsqueeze(0)

        # historical_dynamic_tensor: [1, 270, 6] as you already computed (Has to have the same order as dynamic inputs in the config file)
        x_srad      = spinup_tensor[..., 1:2]
        x_tmax      = spinup_tensor[..., 2:3]
        x_tmin      = spinup_tensor[..., 3:4]
        x_prcp      = spinup_tensor[..., 4:5]
        x_prcp_ch   = spinup_tensor[..., 5:6]
        x_prcp_mswep   = spinup_tensor[..., 6:7]
        x_prcp_gauge   = spinup_tensor[..., 7:8]

        x_srad_norm    = norm_dyn(x_srad,    "srad_W_m2")
        x_tmax_norm    = norm_dyn(x_tmax,    "tmax_C")
        x_tmin_norm    = norm_dyn(x_tmin,    "tmin_C")
        x_prcp_norm    = norm_dyn(x_prcp,    "prcp_mm_day")
        x_prcp_ch_norm = norm_dyn(x_prcp_ch, "prcp_chirps_mm_day")
        x_prcp_mswep_norm    = norm_dyn(x_prcp_mswep,    "prcp_mswep_mm_day")
        x_prcp_gauge_norm = norm_dyn(x_prcp_gauge, "prcp_gauge_mm_day")

        spinup_inputs = {
            "x_d": {
                "srad_W_m2":          x_srad_norm,
                "tmax_C":             x_tmax_norm,
                "tmin_C":             x_tmin_norm,
                "prcp_mm_day":        x_prcp_norm,
                "prcp_chirps_mm_day": x_prcp_ch_norm,
                "prcp_mswep_mm_day":        x_prcp_mswep_norm,
                "prcp_gauge_mm_day":        x_prcp_gauge_norm
            },
            "x_s": static_tensor
        }

        spinup_inputs = {
            "x_d": {k: v for k, v in spinup_inputs["x_d"].items()},
            "x_s": spinup_inputs["x_s"]
        }

        with torch.no_grad():
            out = custom_lstm(spinup_inputs)

        # h0 = out["h_n"]
        # c0 = out["c_n"]
        h_complete_run = out["h_n"][:, -1, :]
        c_complete_run = out["c_n"][:, -1, :]
        y_hat = out["y_hat"]

        predictions_dict = {
            "year": [],
            "prediction": []
        }  

        for year in range(1989, 2008):
            start = pd.Timestamp(year=year, month=month, day=1)
            end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
            # print(f"{start}  -->  {end}")

            historical_df = df_dyn.loc[start:end][dynamic_vars]

            historical_df.index = historical_df.index.map(lambda d: d.replace(year=2008))

            historical_tensor = torch.tensor(historical_df.values.astype(np.float32)).unsqueeze(0)

            # historical_dynamic_tensor: [1, 270, 6] as you already computed
            x_srad_h      = historical_tensor[..., 1:2]
            x_tmax_h      = historical_tensor[..., 2:3]
            x_tmin_h      = historical_tensor[..., 3:4]
            x_prcp_h      = historical_tensor[..., 4:5]
            x_prcp_ch_h   = historical_tensor[..., 5:6]
            x_prcp_mswep_h   = historical_tensor[..., 6:7]
            x_prcp_gauge_h   = historical_tensor[..., 7:8]

            x_srad_norm_h    = norm_dyn(x_srad_h,    "srad_W_m2")
            x_tmax_norm_h    = norm_dyn(x_tmax_h,    "tmax_C")
            x_tmin_norm_h    = norm_dyn(x_tmin_h,    "tmin_C")
            x_prcp_norm_h    = norm_dyn(x_prcp_h,    "prcp_mm_day")
            x_prcp_ch_norm_h = norm_dyn(x_prcp_ch_h, "prcp_chirps_mm_day")
            x_prcp_mswep_norm_h    = norm_dyn(x_prcp_mswep_h,    "prcp_mswep_mm_day")
            x_prcp_gauge_norm_h = norm_dyn(x_prcp_gauge_h, "prcp_gauge_mm_day")

            historical_inputs = {
                "x_d": {
                    "srad_W_m2":          x_srad_norm_h,
                    "tmax_C":             x_tmax_norm_h,
                    "tmin_C":             x_tmin_norm_h,
                    "prcp_mm_day":        x_prcp_norm_h,
                    "prcp_chirps_mm_day": x_prcp_ch_norm_h,
                    "prcp_mswep_mm_day":        x_prcp_mswep_norm_h,
                    "prcp_gauge_mm_day":        x_prcp_gauge_norm_h
                },
                "x_s": static_tensor
            }

            historical_inputs = {
                "x_d": {k: v for k, v in historical_inputs["x_d"].items()},
                "x_s": historical_inputs["x_s"]
            }

            with torch.no_grad():
                out = custom_lstm(historical_inputs, h_0=h_complete_run, c_0=c_complete_run) #h_val and c_val come from the spin up phase

            #predictions
            y_hat = out["y_hat"]

            # Denormalize predictions
            q_center = scaler["xarray_feature_center"]["data_vars"]["QObs_mm_d"]["data"]
            q_scale  = scaler["xarray_feature_scale"]["data_vars"]["QObs_mm_d"]["data"]

            q_center = torch.tensor(q_center, dtype=y_hat.dtype, device=y_hat.device)
            q_scale  = torch.tensor(q_scale,  dtype=y_hat.dtype, device=y_hat.device)

            y_hat_denorm = y_hat * q_scale + q_center

            # Store predictions in the dictionary
            predictions_dict["year"].append(year)         
            predictions_dict["prediction"].append(y_hat_denorm[0, :, 0].cpu().numpy())

        obs_3_months = []

        for year in range(1989, 2008):
            start = pd.Timestamp(year=year, month=month, day=1)
            end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
            obs_vals = df_dyn.loc[f"{start}":f"{end}", 'QObs_mm_d'].values
            obs_3_months.append(obs_vals)

        fig, ax = plt.subplots(figsize=(12, 5))

        # start = pd.Timestamp(year=year, month=month, day=1)
        # end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)

        # Create reference index (fixed year 2008)
        ref_start = pd.Timestamp(year=2008, month=month, day=1)
        ref_end = ref_start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
        dates = pd.date_range(start=ref_start, end=ref_end)

        # Find minimum length across all predictions and observations (handles leap years)
        min_len = min(
            min(len(pred) for pred in predictions_dict["prediction"]),
            min(len(obs) for obs in obs_3_months)
        )
        
        # Truncate dates to min_len
        dates = dates[:min_len]

        # Align predictions
        aligned_preds = []
        for pred in predictions_dict["prediction"]:
            aligned_preds.append(pred[:min_len])
        all_preds = np.vstack(aligned_preds)

        # Compute median across years (axis=0)
        median_pred = np.nanmedian(all_preds, axis=0)
        percentile_10 = np.nanpercentile(all_preds, 10, axis=0)
        percentile_90 = np.nanpercentile(all_preds, 90, axis=0)

        # Align observations
        aligned_obs = []
        for obs in obs_3_months:
            aligned_obs.append(obs[:min_len])
        all_obs = np.vstack(aligned_obs)

        median_obs = np.nanmedian(all_obs, axis=0)
        percentile_10_obs = np.nanpercentile(all_obs, 10, axis=0)
        percentile_90_obs = np.nanpercentile(all_obs, 90, axis=0)

        percentile_0_obs = np.nanpercentile(all_obs, 0, axis=0)
        percentile_100_obs = np.nanpercentile(all_obs, 100, axis=0)

        # Shade between 10th and 90th percentile
        ax.fill_between(dates, percentile_10_obs, percentile_90_obs, color="blue", alpha=0.2, label="10th-90th Percentile (Obs)")
        ax.fill_between(dates, percentile_0_obs, percentile_100_obs, color="blue", alpha=0.1, label="Full distribution (Obs)")
        ax.fill_between(dates, percentile_10, percentile_90, color="red", alpha=0.2, label="10th-90th Percentile (Pred)")

        # Plot median
        ax.plot(dates, median_obs, color="blue", label="Median Observed (Obs)", linewidth=2)
        ax.plot(dates, median_pred, color="red", label="Median Prediction (Pred)", linewidth=2)

        ax.set_xlabel("Date")
        ax.set_ylabel("Streamflow (mm/d)")
        ax.set_title(f"Ensemble streamflow prediction - Basin: {basin} - Month: {month}")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))  # 'Oct 01', 'Nov 15', etc.
        # plt.show()

        fig.savefig(esp_hydrographs_output_dir / f"{basin}_month_{month}_hydrograph.png", dpi=300)
        plt.close(fig)

        print(f"Plotted month {month} for basin {basin}")

Plotted month 1 for basin CAMELS_UY_2
Plotted month 2 for basin CAMELS_UY_2
Plotted month 3 for basin CAMELS_UY_2
Plotted month 4 for basin CAMELS_UY_2
Plotted month 5 for basin CAMELS_UY_2
Plotted month 6 for basin CAMELS_UY_2
Plotted month 7 for basin CAMELS_UY_2
Plotted month 8 for basin CAMELS_UY_2
Plotted month 9 for basin CAMELS_UY_2
Plotted month 10 for basin CAMELS_UY_2
Plotted month 11 for basin CAMELS_UY_2
Plotted month 12 for basin CAMELS_UY_2
Plotted month 1 for basin CAMELS_UY_3
Plotted month 2 for basin CAMELS_UY_3
Plotted month 3 for basin CAMELS_UY_3
Plotted month 4 for basin CAMELS_UY_3
Plotted month 5 for basin CAMELS_UY_3
Plotted month 6 for basin CAMELS_UY_3
Plotted month 7 for basin CAMELS_UY_3
Plotted month 8 for basin CAMELS_UY_3
Plotted month 9 for basin CAMELS_UY_3
Plotted month 10 for basin CAMELS_UY_3
Plotted month 11 for basin CAMELS_UY_3
Plotted month 12 for basin CAMELS_UY_3
Plotted month 1 for basin CAMELS_UY_5
Plotted month 2 for basin CAMELS_UY_5
Plotte